# NetKet outputs analysis for plots

Heree the files are loaded from the main.ipynb `.npz` files from `netket_outputs`. Trainingruns all have own dataframe; for observables without `energy` these will be in other dataframes `other_dfs`


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
# get path
OUTPUT_DIR = Path("results/netket_outputs")
pd.set_option("display.max_columns", None)
# config we need for all runs to plot the spin spin, structure factors and dimer dimer corrs
def get_config(z):
    if "config" not in z.files:
        return {}
    try:
        return json.loads(str(z["config"].item()))
    except Exception as e:
        return {"config_error": str(e)}
# this function gets the series of energy, loss, etc for all the runs it has in the dir
def get_series(z):
    skip = {"config", "exact_energy"}
    return {k: np.asarray(z[k], dtype=float).ravel() for k in z.files
            if k not in skip and np.asarray(z[k]).ndim > 0}
# this function checks for the energies if any present in the .npz files
def scalar(z, key, default=np.nan):
    return z[key].item() if key in z.files and np.asarray(z[key]).ndim == 0 else default

def pad_matrix(lists):
    width = max([len(x) for x in lists], default=0)
    out = np.full((len(lists), width), np.nan)
    for i, x in enumerate(lists):
        out[i, :len(x)] = x
    return out

def load_outputs(output_dir=OUTPUT_DIR):
    rows, run_dfs, energies, configs = [], {}, [], []
    other_dfs = {}
    for path in sorted(output_dir.glob("*.npz")):
        with np.load(path, allow_pickle=True) as z:
            data = get_series(z)
            df = pd.DataFrame(data)
            if "energy" not in data:
                other_dfs[path.stem] = df
                continue
            cfg = get_config(z)
            n = len(data["energy"])
            df.insert(0, "step", np.arange(n))
            exact = scalar(z, "exact_energy")
            final = float(data["energy"][-1]) if n else np.nan
            row = {"run": path.stem, "source_file": path.name, **cfg,
                   "n_steps": n, "energy_final": final,
                   "energy_min": float(np.nanmin(data["energy"])) if n else np.nan,
                   "exact_energy": exact}
            if np.isfinite(final) and np.isfinite(exact) and exact != 0:
                row["delta_E_final"] = final - exact
                row["rel_error_final"] = abs(row["delta_E_final"]) / abs(exact)
            rows.append(row)
            run_dfs[path.stem] = df
            energies.append(data["energy"])
            configs.append(cfg)
    results_df = pd.DataFrame(rows)
    if not results_df.empty:
        cols = [c for c in ["J2_ratio", "d", "h", "n_layers", "source_file"] if c in results_df]
        results_df = results_df.sort_values(cols, na_position="last").reset_index(drop=True)
        order = results_df["run"].to_list()
        run_dfs = {name: run_dfs[name] for name in order}
        energies = [run_dfs[name]["energy"].to_numpy() for name in order]
        configs = [rows[[r["run"] for r in rows].index(name)] for name in order]
        configs = [{k: v for k, v in c.items() if k not in {"run", "source_file", "n_steps", "energy_final", "energy_min", "exact_energy", "delta_E_final", "rel_error_final"}} for c in configs]
    energies = np.array(energies, dtype=object)
    configs = np.array(configs, dtype=object)
    parameters_df = pd.json_normalize(configs).replace({None: np.nan})
    parameters = parameters_df.to_numpy(dtype=object)
    return results_df, run_dfs, other_dfs, energies, pad_matrix(energies), configs, parameters_df, parameters

results_df, run_dfs, other_dfs, energies, energy_matrix, configs, parameters_df, parameters = load_outputs()
print(f"{len(run_dfs)} training runs are loaded; {len(other_dfs)} observables with npz-files are in other_dfs")
#display(results_df)
# remove the biased tests and the ones we need for the plot
results_df = results_df[results_df["symmetrize"] != True]
results_df = results_df[results_df["marshall_sign"] != True]
display(results_df)
#only the ones we need for th eplot
results_d_over_h = results_df.loc[[5,7,9, 18,20,25, 45,46, 44]]
display(results_d_over_h)
# for separate plot to check what happens when various params are changed
results_0_8 = results_df[results_df["J2_ratio"] == 0.8]
display(results_0_8)

### Plotting the d/h = 1,2,4 for j2 = 0, 0.4, 0.8

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# Below we use results_d_over_h for the plots
d_over_h = results_d_over_h["d"] / results_d_over_h["h"]
# for 29, 30, 28  d/h = 1,2,4 respectively so add this
d_over_h.loc[[45, 46, 44]] = [1, 2, 4]
# obtain the min energy for each run in results_d_over_h
energies_d_over_h = results_d_over_h["energy_min"]
ratio_j2 = results_d_over_h["J2_ratio"]
# now plot the rel error vs d/h for all the 3 ratios
# first add the exact energy for each ratio
j2_0_8_exact = -6.78879425
# add to 29, 30, 28 for exact energy
results_d_over_h.loc[[45, 46, 44], "exact_energy"] = j2_0_8_exact
rel_error = abs(results_d_over_h["energy_min"] - results_d_over_h["exact_energy"]) / abs(results_d_over_h["exact_energy"])
# now plot rel_error vs d/h for all 3 ratios
# for notebooks for sharper quality and looks
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "text.usetex": False,          
    "font.family": "serif",
    "mathtext.fontset": "cm",      
    "figure.dpi": 180,             
    "savefig.dpi": 900,     
    "axes.linewidth": 1.3,
    "font.size": 18,
    "axes.labelsize": 24,
    "legend.fontsize": 17,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

fig, ax = plt.subplots(figsize=(5, 5))
colors = {0.0: "#6baed6", 0.4: "#2171b5", 0.8: "#08306b"}
markers = {0.0: "s", 0.4: "o", 0.8: "D"}
for ratio in sorted(ratio_j2.unique()):
    mask = ratio_j2 == ratio
    x = d_over_h[mask]
    y = rel_error[mask]
    order = x.argsort()
    
    ax.plot(
        x.iloc[order],
        y.iloc[order],
        color=colors[ratio],
        marker=markers[ratio],
        linewidth=1.4,
        markersize=9,
        markeredgecolor="black",
        markeredgewidth=1.2,
        label=fr"$J_2/J_1 = {ratio:g}$",
    )
ax.set_yscale("log")
ax.set_xlabel(r"$d/h$")
ax.set_ylim(rel_error.min() * 0.7, rel_error.max() * 3)
ax.set_ylabel(r"$\Delta \epsilon$")
ax.set_xticks([1, 2, 4])
ax.yaxis.set_minor_locator(ticker.LogLocator(base=10, subs=range(2, 10)))
ax.tick_params(which="both", direction="in", top=True, right=True)
ax.grid(False)
ax.legend(loc="upper right", frameon=True,fancybox=True, borderpad=0.2,)
ax.text(
    0.05, 0.95,
    r"$h = 4$",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=17,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        edgecolor="0.8",
    ),
)
plt.tight_layout()
# png savefig
plt.savefig("relative_error_d_over_h.png", dpi=600, bbox_inches="tight")
plt.show()

### Make a specific plot for J2/J1=0.8 and let all these models be plotted to see the effect of increasing order of hyperparameters

In [ ]:
# Drop the indiices of the old files
results_0_8 = results_0_8.drop(index=[39, 40, 41, 42], errors="ignore").copy()
# Extract d and h from source_file
extracted = results_0_8["source_file"].astype(str).str.extract(
    r"_d_?(?P<d>\d+)_h_?(?P<h>\d+)"
)
# Check mathes
display(extracted)

# put inside cols of dataframe
results_0_8.loc[:, "d"] = pd.to_numeric(extracted["d"], errors="coerce").astype("Int64")
results_0_8.loc[:, "h"] = pd.to_numeric(extracted["h"], errors="coerce").astype("Int64")
# also add rel error and extact energy
j2_0_8_exact = -6.78879425
results_0_8["exact_energy"] = j2_0_8_exact
results_0_8["rel_error_final"] = abs(results_0_8["energy_min"] - results_0_8["exact_energy"]) / abs(results_0_8["exact_energy"])
display(results_0_8)

In [ ]:
import numpy as np

results_0_8["model_size"] = (
    results_0_8["d"] *
    results_0_8["h"] *
    results_0_8["n_layers"] *
    results_0_8["d_ff"]
)

results_0_8 = results_0_8.sort_values(
    by=["model_size", "d", "h", "n_layers"]
)

err_col = "rel_error_final"
# keep track of the best moddel in terms of the error that goes down from increasing hyper
best_so_far = results_0_8[err_col].cummin().shift(fill_value=np.inf)

results_0_8_trend = results_0_8[results_0_8[err_col] < best_so_far].copy()

fig, ax = plt.subplots(figsize=(6.5, 4.8))

ax.plot(
    range(len(results_0_8_trend)),
    results_0_8_trend[err_col],
    color="#2171b5",
    marker="s",
    markersize=7,
    markeredgecolor="black",
    linewidth=1.6,
)

ax.set_yscale("log")
ax.set_xlabel("Model index")
ax.set_ylabel(r"$\Delta \epsilon$")

labels = [
    fr"$d={int(d)}, h={int(h)}, n_l={int(nl)}$"
    for d, h, nl in zip(
        results_0_8_trend["d"],
        results_0_8_trend["h"],
        results_0_8_trend["n_layers"],
    )
]

ax.set_xticks(range(len(results_0_8_trend)))
ax.set_xticklabels(labels, rotation=35, ha="right")
ax.text(
    0.75, 0.95,
    r"$J_2/J_1=0.8$",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=14,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        edgecolor="0.8",
    ),
)
ax.tick_params(which="both", direction="in", top=True, right=True)
ax.grid(False)
plt.savefig("model_index_vs_rel_error", dpi=600, bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
# below are the additional data for plotting the spin spin etc if anyone would like to prit it specificaly
print("run_dfs keys:", list(run_dfs))
print("energies:", energies.shape, "energy_matrix:", energy_matrix.shape)
print("configs:", configs.shape, "parameters:", parameters.shape)

### Physical observables: Spin-spin correlation, structure factor and dimer- dimer correlation

In [ ]:
def spin_correlation(configs):
    """C(r) = <s_i s_{i+r}> averaged over samples and sites."""
    s = np.asarray(configs, dtype=np.float64)
    L = s.shape[1]
    return np.array([np.mean(s * np.roll(s, -r, axis=1)) for r in range(L)])

def structure_factor_from_C(C):
    """S(k) from the spin correlation C(r)."""
    C = np.asarray(C)
    return np.fft.fft(C).real

def dimer_correlation(configs):
    """D(r) = <B_i B_{i+r}> with B_i = s_i s_{i+1}."""
    s = np.asarray(configs, dtype=np.float64)
    B = s * np.roll(s, -1, axis=1)
    L = s.shape[1]
    return np.array([np.mean(B * np.roll(B, -r, axis=1)) for r in range(L)])



In [ ]:
import matplotlib.pyplot as plt

# choose a specific run one would like the plot the observables for for example d/h = 3.
run1 = "jax_vit_j1j2_L16_J2ratio_0.4_d_12_h_4_nl_2"
run2 = "jax_vit"
print(run1)
obs_key = run1.replace("jax_vit_j1j2", "jax_vit_observables")

print("obs_key:", obs_key)
print("available:", obs_key in other_dfs)

obs = other_dfs[obs_key]
display(obs.head())

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

axes[0].plot(obs["r"], obs["C"], color="purple", marker="o", markersize=3)
axes[0].set_xlabel("r")
axes[0].set_ylabel("C(r)")
axes[0].set_title("Spin-spin correlation")
axes[0].grid(True, alpha=0.3)

axes[1].plot(obs["k"], obs["S"], color="green", marker="o", markersize=3)
axes[1].set_xlabel("k")
axes[1].set_ylabel("S(k)")
axes[1].set_title("Structure factor")
axes[1].grid(True, alpha=0.3)

axes[2].plot(obs["r"], obs["D"], color = "cyan", marker="o", markersize=3)
axes[2].set_xlabel("r")
axes[2].set_ylabel("D(r)")
axes[2].set_title("Dimer-dimer correlation")
axes[2].grid(True, alpha=0.3)

fig.suptitle(r"$\frac{J_2}{J_1} = 0.4$, $\frac{d}{h} = 3$, h = 4")
fig.tight_layout()
plt.show()

In [ ]:
%config InlineBackend.figure_format = "retina"
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "figure.dpi": 180,
    "savefig.dpi": 600,
    "axes.linewidth": 1.2,
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.titlesize": 15,
    "legend.fontsize": 12,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

runs = {
    0.0: "jax_vit_j1j2_L16_J2ratio_0.0_h4d8",
    0.4: "jax_vit_j1j2_L16_J2ratio_0.4_d_8_h_4_nl_2",
    0.8: "jax_jit_observables_L16_J2ratio_0.8_d_8_h_4_nl_6"
}

colors = {
    0.0: "#6baed6",
    0.4: "#2171b5",
    0.8: "#08306b",
}

colors2 = {
    0.0: "#a1d99b",
    0.4: "#31a354",
    0.8: "#006d2c",
}

colors3 = {
    0.0: "#fc9272",
    0.4: "#de2d26",
    0.8: "#99000d",
}

def get_obs(run):
    obs_key = run.replace("jax_vit_j1j2", "jax_vit_observables")
    print(obs_key, obs_key in other_dfs)
    return other_dfs[obs_key]

def plot_observable_vertical(y_col, x_col, ylabel, xlabel, title, filename, color_set):
    fig, axes = plt.subplots(
        len(runs), 1,
        figsize=(5.2, 2.45 * len(runs)),
        sharex=True
    )

    if len(runs) == 1:
        axes = [axes]

    for ax, (ratio, run) in zip(axes, runs.items()):
        obs = get_obs(run).copy()

        # Convert raw dimer-dimer correlation to connected spin-operator form
        # Only applied when plotting D
        if y_col == "D_connected":
            C_raw_r1 = obs.loc[obs["r"] == 1, "C"].iloc[0]
            obs["D_connected"] = obs["D"] / 16 - (C_raw_r1 / 4)**2

        # For structure factor restrict k to [0, pi]
        if x_col == "k":
            obs = obs[(obs[x_col] >= 0) & (obs[x_col] <= np.pi)]

        ax.plot(
            obs[x_col],
            obs[y_col],
            color=color_set[ratio],
            marker="o",
            markersize=4.5,
            markeredgecolor="black",
            markeredgewidth=0.7,
            linewidth=1.3,
        )

        ax.set_title(fr"$J_2/J_1 = {ratio:g}$", pad=5)
        ax.set_ylabel(ylabel)
        ax.tick_params(which="both", direction="in", top=True, right=True)
        ax.grid(False)

        if x_col == "k":
            ax.set_xlim(0, np.pi)
            ax.set_xticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
            ax.set_xticklabels([
                r"$0$",
                r"$\pi/4$",
                r"$\pi/2$",
                r"$3\pi/4$",
                r"$\pi$"
            ])

    axes[-1].set_xlabel(xlabel)

    fig.suptitle(title, y=1.01, fontsize=16)
    fig.tight_layout()
    plt.savefig(filename + ".png", dpi=600, bbox_inches="tight")
    plt.show()

## Plot to check below all the phys obserbavles and also use the correct dimer dimer corr func in plot

In [ ]:
plot_observable_vertical(
    "C", "r",
    r"$C(r)$",
    r"$r$",
    r"Spin-spin correlation, $d/h = 2$, $d = 8$",
    "spin_spin_correlation_dh2_d8_vertical",
    colors3
)

plot_observable_vertical(
    "S", "k",
    r"$S(k)$",
    r"$k$",
    r"Structure factor, $d/h = 2$, $d = 8$",
    "structure_factor_dh2_d8_vertical",
    colors2
)

plot_observable_vertical(
    "D_connected", "r",
    r"$D(r)$",
    r"$r$",
    r"Connected dimer-dimer correlation, $d/h = 2$, $d = 8$",
    "dimer_dimer_correlation_connected_dh2_d8_vertical",
    colors
)